In [ ]:
!pip install -q torch transformers accelerate bitsandbytes sentence-transformers unstructured[all-docs] langchain chromadb langchain_community

In [ ]:
!mkdir -p "./documents"
!wget https://www.gov.nl.ca/ecc/files/env-protection-pesticides-business-manuals-applic-chapter7.pdf -O "./documents/env-protection-pesticides-business-manuals-applic-chapter7.pdf"
!wget https://ipm.ifas.ufl.edu/pdfs/Citrus_IPM_090913.pptx -O "./documents/Citrus_IPM_090913.pptx"
!wget https://www.gutenberg.org/ebooks/45957.epub3.images -O "./documents/45957.epub"
!wget https://blog.fifthroom.com/what-to-do-about-harmful-garden-and-plant-insects-and-pests.html -O "./documents/what-to-do-about-harmful-garden-and-plant-insects-and-pests.html"

In [ ]:
# Optional cell to reduce the amount of logs

import logging

logger = logging.getLogger("unstructured.ingest")
logger.root.removeHandler(logger.root.handlers[0])

In [ ]:
import os

from unstructured.ingest.connector.local import SimpleLocalConfig
from unstructured.ingest.interfaces import PartitionConfig, ProcessorConfig, ReadConfig
from unstructured.ingest.runner import LocalRunner

output_path = "./local-ingest-output"

runner = LocalRunner(
    processor_config=ProcessorConfig(
        # logs verbosity
        verbose=True,
        # the local directory to store outputs
        output_dir=output_path,
        num_processes=2,
        ),
    read_config=ReadConfig(),
    partition_config=PartitionConfig(
        partition_by_api=True,
        api_key="YOUR_UNSTRUCTURED_API_KEY",
        ),
    connector_config=SimpleLocalConfig(
        input_path="./documents",
        # whether to get the documents recursively from given directory
        recursive=False,
        ),
    )
runner.run()


In [ ]:
from unstructured.staging.base import elements_from_json

elements = []

for filename in os.listdir(output_path):
    filepath = os.path.join(output_path, filename)
    elements.extend(elements_from_json(filepath))

In [ ]:
from unstructured.chunking.title import chunk_by_title

chunked_elements = chunk_by_title(elements,
                                  # maximum for chunk size
                                  max_characters=512,
                                  # You can choose to combine consecutive elements that are too small
                                  # e.g. individual list items
                                  combine_text_under_n_chars=200,
                                  )


In [ ]:
from langchain_core.documents import Document

documents = []
for chunked_element in chunked_elements:
    metadata = chunked_element.metadata.to_dict()
    metadata["source"] = metadata["filename"]
    del metadata["languages"]
    documents.append(Document(page_content=chunked_element.text, metadata=metadata))

## Setting up the retriever

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain.embeddings import HuggingFaceEmbeddings

from langchain.vectorstores import utils as chromautils

# ChromaDB doesn't support complex metadata, e.g. lists, so we drop it here.
# If you're using a different vector store, you may not need to do this
docs = chromautils.filter_complex_metadata(documents)

embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-base-en-v1.5")
vectorstore = Chroma.from_documents(documents, embeddings)
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
from langchain.prompts import PromptTemplate
from langchain.llms import HuggingFacePipeline
from transformers import pipeline
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from langchain.chains import RetrievalQA

In [ ]:
model_name = "meta-llama/Meta-Llama-3-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config)
tokenizer = AutoTokenizer.from_pretrained(model_name)

terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

text_generation_pipeline = pipeline(
    model=model,
    tokenizer=tokenizer,
    task="text-generation",
    temperature=0.2,
    do_sample=True,
    repetition_penalty=1.1,
    return_full_text=False,
    max_new_tokens=200,
    eos_token_id=terminators,
)

llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

prompt_template = """
<|start_header_id|>user<|end_header_id|>
You are an assistant for answering questions using provided context.
You are given the extracted parts of a long document and a question. Provide a conversational answer.
If you don't know the answer, just say "I do not know." Don't make up an answer.
Question: {question}
Context: {context}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=prompt_template,
)


qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=retriever,
    chain_type_kwargs={"prompt": prompt}
)

In [ ]:
question = "Are aphids a pest?"

qa_chain.invoke(question)['result']

Output:

```bash
Yes, aphids are considered pests because they feed on the nutrient-rich liquids within plants, causing damage and potentially spreading disease. In fact, they're known to multiply quickly, which is why it's essential to control them promptly. As mentioned in the text, aphids can also attract ants, which are attracted to the sweet, sticky substance they produce called honeydew. So, yes, aphids are indeed a pest that requires attention to prevent further harm to your plants!
```

In [1]:
!pip install -q torch transformers accelerate bitsandbytes sentence-transformers faiss-gpu langchain langchain_community pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 MB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.5/85.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 298.0/298.0 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 kB 4.7 MB/s eta 0:00:00


In [3]:
!pip install -q torch transformers accelerate bitsandbytes sentence-transformers faiss-gpu langchain langchain_community python-pptx pypdf python-docx beautifulsoup4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.3/244.3 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.9/159.9 kB 13.4 MB/s eta 0:00:00


In [6]:
!huggingface-cli login




    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) Y
Token is valid (permission: read).
The token `read` has been saved to /root/.cache/huggingface/stored_tokens
Cannot authenticate through git-credential as no helper is defined on your machine.
You might have to re-authenticate when pushi

شغال راج مع pdf ,وملفات مختلفة الامتدادات رائع

In [7]:
#!pip install -q torch transformers accelerate bitsandbytes sentence-transformers faiss-gpu langchain langchain_community python-pptx pypdf python-docx beautifulsoup4 EbookLib

import os
from typing import List
from langchain_community.document_loaders import (
    PyPDFLoader,
    BSHTMLLoader
)
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from pptx import Presentation
import ebooklib
from ebooklib import epub
from bs4 import BeautifulSoup

def get_pptx_text(path: str) -> str:
    prs = Presentation(path)
    text = []
    for slide in prs.slides:
        for shape in slide.shapes:
            if hasattr(shape, "text"):
                text.append(shape.text)
    return "\n".join(text)

def get_epub_text(path: str) -> str:
    try:
        book = epub.read_epub(path)
        texts = []
        for item in book.get_items():
            if item.get_type() == ebooklib.ITEM_DOCUMENT:
                content = item.get_content().decode('utf-8')
                soup = BeautifulSoup(content, 'html.parser')
                texts.append(soup.get_text())
        return "\n".join(texts)
    except Exception as e:
        print(f"Error processing EPUB: {str(e)}")
        return ""

def load_documents(directory: str) -> List[Document]:
    documents = []
    for filename in os.listdir(directory):
        filepath = os.path.join(directory, filename)
        try:
            if filename.endswith('.pdf'):
                loader = PyPDFLoader(filepath)
                documents.extend(loader.load())
            elif filename.endswith('.pptx'):
                text = get_pptx_text(filepath)
                documents.append(Document(page_content=text, metadata={"source": filepath}))
            elif filename.endswith('.epub'):
                text = get_epub_text(filepath)
                if text:  # Only add if we got some text
                    documents.append(Document(page_content=text, metadata={"source": filepath}))
            elif filename.endswith('.html'):
                loader = BSHTMLLoader(filepath)
                documents.extend(loader.load())
        except Exception as e:
            print(f"Error loading {filename}: {str(e)}")
            continue

    return documents

# Create documents directory and download files
!mkdir -p "./documents"
!wget https://www.gov.nl.ca/ecc/files/env-protection-pesticides-business-manuals-applic-chapter7.pdf -O "./documents/env-protection-pesticides-business-manuals-applic-chapter7.pdf"
!wget https://ipm.ifas.ufl.edu/pdfs/Citrus_IPM_090913.pptx -O "./documents/Citrus_IPM_090913.pptx"
!wget https://www.gutenberg.org/ebooks/45957.epub3.images -O "./documents/45957.epub"
!wget https://blog.fifthroom.com/what-to-do-about-harmful-garden-and-plant-insects-and-pests.html -O "./documents/what-to-do-about-harmful-garden-and-plant-insects-and-pests.html"

# Load documents
print("Loading documents...")
documents = load_documents("./documents")
print(f"Loaded {len(documents)} documents")

# Split documents into chunks
print("Splitting documents into chunks...")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", "!", "?", ",", " ", ""],
    length_function=len
)
chunks = text_splitter.split_documents(documents)
print(f"Created {len(chunks)} chunks")

# Initialize embeddings and vector store
print("Initializing embeddings and vector store...")
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-base-en-v1.5")
vectorstore = FAISS.from_documents(chunks, embeddings)

# Save the FAISS index (optional)
vectorstore.save_local("faiss_index")

# Create retriever
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})

# Set up the language model
print("Setting up the language model...")
from huggingface_hub import notebook_login
notebook_login()

from langchain.prompts import PromptTemplate
from langchain.llms import HuggingFacePipeline
from transformers import pipeline
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from langchain.chains import RetrievalQA

model_name = "meta-llama/Meta-Llama-3-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config)
tokenizer = AutoTokenizer.from_pretrained(model_name)

terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

text_generation_pipeline = pipeline(
    model=model,
    tokenizer=tokenizer,
    task="text-generation",
    temperature=0.2,
    do_sample=True,
    repetition_penalty=1.1,
    return_full_text=False,
    max_new_tokens=200,
    eos_token_id=terminators,
)

llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

prompt_template = """
<|start_header_id|>user<|end_header_id|>
You are an assistant for answering questions using provided context.
You are given the extracted parts of a long document and a question. Provide a conversational answer.
If you don't know the answer, just say "I do not know." Don't make up an answer.
Question: {question}
Context: {context}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=prompt_template,
)

qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=retriever,
    chain_type_kwargs={"prompt": prompt}
)

print("Setup complete! Ready to answer questions.")

# Example usage
question = "Are aphids a pest?"
result = qa_chain.invoke(question)['result']
print("\nQuestion:", question)
print("Answer:", result)

--2025-01-13 20:26:20--  https://www.gov.nl.ca/ecc/files/env-protection-pesticides-business-manuals-applic-chapter7.pdf
Resolving www.gov.nl.ca (www.gov.nl.ca)... 98.143.128.70
Connecting to www.gov.nl.ca (www.gov.nl.ca)|98.143.128.70|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1914250 (1.8M) [application/pdf]
Saving to: ‘./documents/env-protection-pesticides-business-manuals-applic-chapter7.pdf’

./documents/env-pro 100%[===================>]   1.83M  1.70MB/s    in 1.1s    

2025-01-13 20:26:21 (1.70 MB/s) - ‘./documents/env-protection-pesticides-business-manuals-applic-chapter7.pdf’ saved [1914250/1914250]

--2025-01-13 20:26:21--  https://ipm.ifas.ufl.edu/pdfs/Citrus_IPM_090913.pptx
Resolving ipm.ifas.ufl.edu (ipm.ifas.ufl.edu)... 128.227.68.231
Connecting to ipm.ifas.ufl.edu (ipm.ifas.ufl.edu)|128.227.68.231|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4248570 (4.1M) [application/vnd.openxmlformats-officedocument.presentat

/usr/local/lib/python3.10/dist-packages/ebooklib/epub.py:1395: UserWarning: In the future version we will turn default option ignore_ncx to True.
  warnings.warn('In the future version we will turn default option ignore_ncx to True.')
/usr/local/lib/python3.10/dist-packages/ebooklib/epub.py:1423: FutureWarning: This search incorrectly ignores the root element, and will be fixed in a future version.  If you rely on the current behaviour, change it to './/xmlns:rootfile[@media-type]'
  for root_file in tree.findall('//xmlns:rootfile[@media-type]', namespaces={'xmlns': NAMESPACES['CONTAINERNS']}):


Loaded 31 documents
Splitting documents into chunks...
Created 658 chunks
Initializing embeddings and vector store...
Setting up the language model...


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

`low_cpu_mem_usage` was None, now default to True since model is quantized.


...

tokenizer_config.json:   0%|          | 0.00/51.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

Device set to use cuda:0
<ipython-input-7-21c95ea3a8b1>:141: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=text_generation_pipeline)
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


Setup complete! Ready to answer questions.

Question: Are aphids a pest?
Answer: Aphids are indeed considered pests! As mentioned in the context, they feed on the nutrient-rich liquids within plants, multiplying quickly and causing damage to the plant's tissues. In fact, aphids can even deform the leaves, stunt the growth, or cause them to curl. And, as you've noticed, ants often arrive on the scene before you spot the aphids themselves, attracted to the sweet, sticky residue left behind - which we lovingly refer to as "aphid drool"! So, yes, aphids are most definitely considered pests that require control measures to prevent them from wreaking havoc on your beautiful plants.


كود محسن

شغال رائع

In [1]:
# Requirements installation
#!pip install -q torch transformers accelerate bitsandbytes sentence-transformers faiss-gpu langchain langchain_community python-pptx pypdf python-docx beautifulsoup4 EbookLib tqdm

import os
from typing import List, Dict, Optional
from langchain_community.document_loaders import PyPDFLoader, BSHTMLLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from pptx import Presentation
import ebooklib
from ebooklib import epub
from bs4 import BeautifulSoup
from tqdm import tqdm
import logging
import json
from datetime import datetime

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('qa_system.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

class DocumentProcessor:
    def __init__(self, chunk_size: int = 512, chunk_overlap: int = 50):
        """Initialize the document processor with configurable parameters."""
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.supported_formats = {'.pdf', '.pptx', '.epub', '.html'}

    def get_pptx_text(self, path: str) -> str:
        """Extract text from PowerPoint files."""
        try:
            prs = Presentation(path)
            text = []
            for slide in prs.slides:
                slide_text = []
                for shape in slide.shapes:
                    if hasattr(shape, "text"):
                        slide_text.append(shape.text.strip())
                text.append(" ".join(filter(None, slide_text)))
            return "\n\n".join(filter(None, text))
        except Exception as e:
            logger.error(f"Error processing PPTX {path}: {str(e)}")
            return ""

    def get_epub_text(self, path: str) -> str:
        """Extract text from EPUB files."""
        try:
            book = epub.read_epub(path)
            texts = []
            for item in book.get_items():
                if item.get_type() == ebooklib.ITEM_DOCUMENT:
                    content = item.get_content().decode('utf-8')
                    soup = BeautifulSoup(content, 'html.parser')
                    texts.append(soup.get_text().strip())
            return "\n\n".join(filter(None, texts))
        except Exception as e:
            logger.error(f"Error processing EPUB {path}: {str(e)}")
            return ""

    def load_documents(self, directory: str) -> List[Document]:
        """Load documents from the specified directory."""
        documents = []
        files = [f for f in os.listdir(directory) if os.path.splitext(f)[1].lower() in self.supported_formats]

        for filename in tqdm(files, desc="Loading documents"):
            filepath = os.path.join(directory, filename)
            try:
                if filename.endswith('.pdf'):
                    loader = PyPDFLoader(filepath)
                    documents.extend(loader.load())
                elif filename.endswith('.pptx'):
                    text = self.get_pptx_text(filepath)
                    if text:
                        documents.append(Document(page_content=text, metadata={
                            "source": filepath,
                            "type": "pptx",
                            "processed_date": datetime.now().isoformat()
                        }))
                elif filename.endswith('.epub'):
                    text = self.get_epub_text(filepath)
                    if text:
                        documents.append(Document(page_content=text, metadata={
                            "source": filepath,
                            "type": "epub",
                            "processed_date": datetime.now().isoformat()
                        }))
                elif filename.endswith('.html'):
                    loader = BSHTMLLoader(filepath)
                    documents.extend(loader.load())
            except Exception as e:
                logger.error(f"Error loading {filename}: {str(e)}")
                continue

        return documents

    def process_documents(self, directory: str) -> List[Document]:
        """Process documents and split into chunks."""
        logger.info("Starting document processing")

        # Load documents
        documents = self.load_documents(directory)
        logger.info(f"Loaded {len(documents)} documents")

        # Split documents
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=self.chunk_size,
            chunk_overlap=self.chunk_overlap,
            separators=["\n\n", "\n", ".", "!", "?", ",", " ", ""],
            length_function=len
        )
        chunks = splitter.split_documents(documents)
        logger.info(f"Created {len(chunks)} chunks")

        return chunks

class QASystem:
    def __init__(self, model_name: str = "meta-llama/Meta-Llama-3-8B-Instruct",
                 embedding_model: str = "BAAI/bge-base-en-v1.5",
                 save_dir: str = "./qa_system"):
        """Initialize the QA system."""
        self.model_name = model_name
        self.embedding_model = embedding_model
        self.save_dir = save_dir
        os.makedirs(save_dir, exist_ok=True)

        self.setup_model()
        logger.info("QA System initialized successfully")

    def setup_model(self):
        """Set up the language model and embeddings."""
        from langchain.embeddings import HuggingFaceEmbeddings
        from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
        import torch

        # Setup embeddings
        self.embeddings = HuggingFaceEmbeddings(model_name=self.embedding_model)

        # Setup language model
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16
        )

        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_name,
            quantization_config=bnb_config
        )
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)

    def build_index(self, chunks: List[Document]):
        """Build the FAISS index from document chunks."""
        from langchain_community.vectorstores import FAISS

        self.vectorstore = FAISS.from_documents(chunks, self.embeddings)
        self.vectorstore.save_local(os.path.join(self.save_dir, "faiss_index"))
        logger.info("FAISS index built and saved successfully")

    def load_index(self):
        """Load existing FAISS index."""
        from langchain_community.vectorstores import FAISS

        index_path = os.path.join(self.save_dir, "faiss_index")
        if os.path.exists(index_path):
            self.vectorstore = FAISS.load_local(index_path, self.embeddings)
            logger.info("FAISS index loaded successfully")
            return True
        return False

    def setup_chain(self):
        """Set up the QA chain."""
        from langchain.chains import RetrievalQA
        from langchain.prompts import PromptTemplate
        from langchain.llms import HuggingFacePipeline
        from transformers import pipeline

        # Setup pipeline
        text_generation_pipeline = pipeline(
            model=self.model,
            tokenizer=self.tokenizer,
            task="text-generation",
            temperature=0.2,
            do_sample=True,
            repetition_penalty=1.1,
            return_full_text=False,
            max_new_tokens=200,
            eos_token_id=[
                self.tokenizer.eos_token_id,
                self.tokenizer.convert_tokens_to_ids("<|eot_id|>")
            ]
        )

        llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

        # Setup prompt
        prompt_template = """
        <|start_header_id|>user<|end_header_id|>
        You are an assistant for answering questions using provided context.
        You are given the extracted parts of a long document and a question. Provide a conversational answer.
        If you don't know the answer, just say "I do not know." Don't make up an answer.
        Question: {question}
        Context: {context}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
        """

        prompt = PromptTemplate(
            input_variables=["context", "question"],
            template=prompt_template
        )

        # Create retriever
        retriever = self.vectorstore.as_retriever(
            search_type="similarity",
            search_kwargs={"k": 3}
        )

        # Setup QA chain
        self.qa_chain = RetrievalQA.from_chain_type(
            llm,
            retriever=retriever,
            chain_type_kwargs={"prompt": prompt}
        )
        logger.info("QA chain setup completed")

    def answer_question(self, question: str) -> Dict:
        """Answer a question using the QA chain."""
        try:
            result = self.qa_chain.invoke(question)
            return {
                "question": question,
                "answer": result['result'],
                "timestamp": datetime.now().isoformat()
            }
        except Exception as e:
            logger.error(f"Error answering question: {str(e)}")
            return {
                "question": question,
                "error": str(e),
                "timestamp": datetime.now().isoformat()
            }

# Usage example
def main():
    # Initialize document processor
    processor = DocumentProcessor(chunk_size=512, chunk_overlap=50)

    # Create directories and download files if needed
    os.makedirs("./documents", exist_ok=True)
    # [Download files code here...]

    # Process documents
    chunks = processor.process_documents("./documents")

    # Initialize QA system
    qa_system = QASystem()

    # Try to load existing index, or build new one
    if not qa_system.load_index():
        qa_system.build_index(chunks)

    # Setup QA chain
    qa_system.setup_chain()

    # Example question
    question = "Are aphids a pest?"
    result = qa_system.answer_question(question)
    print("\nQuestion:", result["question"])
    print("Answer:", result["answer"])
    print("Timestamp:", result["timestamp"])

if __name__ == "__main__":
    main()

Loading documents:   0%|          | 0/4 [00:00<?, ?it/s]/usr/local/lib/python3.10/dist-packages/ebooklib/epub.py:1395: UserWarning: In the future version we will turn default option ignore_ncx to True.
  warnings.warn('In the future version we will turn default option ignore_ncx to True.')
/usr/local/lib/python3.10/dist-packages/ebooklib/epub.py:1423: FutureWarning: This search incorrectly ignores the root element, and will be fixed in a future version.  If you rely on the current behaviour, change it to './/xmlns:rootfile[@media-type]'
  for root_file in tree.findall('//xmlns:rootfile[@media-type]', namespaces={'xmlns': NAMESPACES['CONTAINERNS']}):
Loading documents: 100%|██████████| 4/4 [00:03<00:00,  1.27it/s]
<ipython-input-1-4e17738eb795>:143: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To 

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0
<ipython-input-1-4e17738eb795>:201: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=text_generation_pipeline)
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.



Question: Are aphids a pest?
Answer:  Ah, yes! According to the context, aphids are indeed considered pests. In fact, they're described as "small, soft-bodied plant-lice" that feed on the nutrient-rich liquids within plants, causing damage and potentially destroying them. The text also mentions how aphids multiply quickly and need to be controlled immediately to prevent harm to the plants. So, to answer your question, yes, aphids are most definitely considered pests!
Timestamp: 2025-01-13T20:39:54.517678
